# 图像分类

In [ ]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

In [ ]:
trans = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(
    root="../data", train=True, transform=trans, download=True
)
mnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, transform=trans, download=True
)
mnist_train.data.shape, mnist_test.data.shape

In [ ]:
mnist_train.targets.shape, mnist_test.targets.shape


In [ ]:
mnist_test.targets.dtype

In [ ]:
# 将label编码转换为名称
def get_fashion_mnist_labels(labels): 
    """返回Fashion-MNIST数据集的文本标签""" 
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',  'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot'] 
    return [text_labels[int(i)] for i in labels]

In [ ]:
def net(X: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.add(torch.matmul(X,w), b)

def softmax(Y: torch.Tensor) -> torch.Tensor:
    Y = Y - Y.max(dim=1, keepdim=True)[0]
    yexp = torch.exp(Y)
    sum_exp = yexp.sum(dim=1, keepdim=True)
    return yexp / sum_exp
def onehot(targets: torch.Tensor, num_labels: int) -> torch.Tensor:
    h = torch.arange(0, num_labels, dtype=targets.dtype)
    b = targets.reshape((-1,1))==h
    m = torch.zeros(b.shape)
    m[b] = 1.0
    return m
def lossFunction(Y: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    return -(T * torch.log(Y+1e-9)).sum()

In [ ]:
num_labels = 10
num_epochs = 10000
batch_size = 1000
rate = 0.1

train_data = mnist_train.data.flatten(start_dim=1).float() / 255 # 图片展平，像素值0-255 归一化
train_targets = onehot(mnist_train.targets, num_labels)
train_size = train_data.shape[0]
num_features = train_data.shape[1]


In [ ]:
w = torch.normal(0, 1, (num_features, num_labels)) / (num_features ** 0.5) # 初始值不能太大
w.requires_grad_(True)
b = torch.zeros(num_labels)
b.requires_grad_(True)

def lossFunction_mean(Y: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    return -(T * torch.log(Y+1e-9)).sum(dim=1).mean()

losses = []
for epoch in range(num_epochs):
    lb = 0
    epoch_loss = 0
    batches = 0
    while lb < train_size:
        rb = min(lb+batch_size, train_size)
        nb = rb - lb
        if w.grad is not None: w.grad.zero_()
        if b.grad is not None: b.grad.zero_()
        X = train_data[lb:rb,:]
        T = train_targets[lb:rb,:]
        Y = softmax(net(X,w,b))
        loss = lossFunction_mean(Y,T)
        loss.backward()
        with torch.no_grad():
            w.sub_(w.grad * rate)
            b.sub_(b.grad * rate)
        epoch_loss += loss.item()
        batches += 1
        lb += batch_size

    avg_loss = epoch_loss / batches
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {avg_loss:.6f}, w.grad mean = {w.grad.abs().mean():.6f}")

In [ ]:
def pred(data: torch.Tensor, w: torch.Tensor, b: torch.Tensor):
    data = data.flatten(start_dim=1).float() / 255.0
    with torch.no_grad():
        Y = softmax(net(data, w, b))
        L = Y.argmax(dim=1)
        return L

pred_targets = pred(mnist_test.data, w, b)
num_pred_right = (mnist_test.targets == pred_targets).sum()
num_test_samples = mnist_test.data.shape[0]
accuracy = num_pred_right * 1.0 / num_test_samples
accuracy